# Advanced 12 — Asymptotic Likelihood Theory

Practice notebook for the **"Asymptotic Likelihood Theory"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Consistency of the MLE

Under standard regularity conditions (identifiability, differentiability,
integrability), the maximum likelihood estimator \(\hat{\theta}_n\) of a
scalar parameter \(\theta_0\) is **consistent**:

$$
\hat{\theta}_n \;\to\; \theta_0 \qquad (n\to\infty).
$$

Consistency says: as the sample grows, the MLE collapses onto the true
parameter. We will *see* this happen by simulating many datasets of
increasing size from a known model, computing the MLE for each, and
watching the histogram of \(\hat{\theta}_n\) concentrate at \(\theta_0\).

We use a **Poisson\((\theta_0)\)** model, where the MLE is the sample mean
\(\hat{\theta}_n = \bar{X}\) and the true rate is \(\theta_0 = 3\). The Fisher
information for one observation is \(I(\theta) = 1/\theta\), so the asymptotic
variance is \(\theta_0/n\) — a useful reference for the next part.


In [ ]:
def poisson_mle(sample):
    '''MLE of the Poisson rate theta is the sample mean.'''
    return np.mean(sample)

theta0 = 3.0           # true Poisson rate
reps = 2000            # number of independent datasets per n
ns = [5, 20, 80, 320]  # increasing sample sizes

rng = np.random.default_rng(42)
fig, axes = plt.subplots(1, len(ns), figsize=(16, 4.5), sharey=False)
for ax, n in zip(axes, ns):
    # reps datasets, each of size n  ->  shape (reps, n)
    data = rng.poisson(theta0, size=(reps, n))
    mles = poisson_mle(data.mean(axis=1))
    ax.hist(mles, bins=40, density=True, color="steelblue",
            edgecolor="white", alpha=0.85)
    ax.axvline(theta0, color="crimson", lw=2, label=rf"$\theta_0={theta0}$")
    ax.set_title(f"n = {n}")
    ax.set_xlabel(r"$\hat{\theta}_n$")
    if n == ns[0]:
        ax.set_ylabel("density")
    ax.legend(fontsize=8)
fig.suptitle(rf"Poisson MLE $\hat{{\theta}}_n=\bar{{X}}$ concentrates at $\theta_0={theta0}$ as $n$ grows",
             y=1.02)
plt.tight_layout()
plt.show()

# Numerical check: variance of the MLE should shrink like theta0 / n
print(f"{'n':>6} {'mean(MLE)':>10} {'var(MLE)':>10} {'theta0/n':>10}")
for n in ns:
    data = rng.poisson(theta0, size=(reps, n))
    mles = data.mean(axis=1)
    print(f"{n:>6} {mles.mean():>10.4f} {mles.var(ddof=1):>10.4f} {theta0/n:>10.4f}")


---
## Part 2 — Asymptotic normality of the MLE

Consistency alone is not enough for inference — we need a *distributional*
limit. Under the same regularity conditions the MLE is **asymptotically
normal**:

$$
\sqrt{n}\,(\hat{\theta}_n - \theta_0)
\;\xRightarrow{d}\;
\mathcal{N}\!\left(0,\ \frac{1}{I(\theta_0)}\right),
$$

where \(I(\theta_0)\) is the **Fisher information for one observation**.
Equivalently,

$$
\hat{\theta}_n \;\overset{\cdot}{\sim}\;
\mathcal{N}\!\left(\theta_0,\ \frac{1}{n\,I(\theta_0)}\right).
$$

For the Poisson model, \(I(\theta)=1/\theta\), so the limit law is
\(\mathcal{N}\!\big(0,\theta_0\big)\) for the scaled quantity
\(\sqrt{n}(\hat{\theta}_n-\theta_0)\). We standardize the simulated MLEs
with the *true* \(\theta_0\) (so we are checking the theorem, not
estimating) and overlay the limiting normal density on the histogram.


In [ ]:
theta0 = 3.0
n = 80
reps = 50000
rng = np.random.default_rng(7)

# Simulate many datasets, compute sqrt(n)(theta_hat - theta0)
data = rng.poisson(theta0, size=(reps, n))
theta_hat = data.mean(axis=1)
scaled = np.sqrt(n) * (theta_hat - theta0)   # should -> N(0, theta0)

# Asymptotic variance is 1 / I(theta0) = theta0
limit_var = 1.0 / (1.0 / theta0)            # = theta0
limit_sd = np.sqrt(limit_var)

print(f"n = {n}, reps = {reps}")
print(f"empirical mean of sqrt(n)(theta_hat-theta0) = {scaled.mean():+.4f} (should be ~0)")
print(f"empirical variance                        = {scaled.var(ddof=1):.4f} (limit {limit_var:.4f})")
print(f"limit sd = sqrt(1/I(theta0)) = sqrt(theta0) = {limit_sd:.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(scaled, bins=80, density=True, color="seagreen",
        edgecolor="white", alpha=0.75, label=r"simulated $\sqrt{n}(\hat{\\theta}_n-\\theta_0)$")
grid = np.linspace(scaled.min(), scaled.max(), 300)
ax.plot(grid, stats.norm.pdf(grid, loc=0.0, scale=limit_sd),
        color="crimson", lw=2.5,
        label=rf"$\mathcal{{N}}(0,\,1/I(\theta_0))=\mathcal{{N}}(0,\,{limit_var:.1f})$")
ax.axvline(0, color="black", lw=1, ls="--")
ax.set_xlabel(r"$\sqrt{n}(\hat{\theta}_n-\theta_0)$")
ax.set_ylabel("density")
ax.set_title(rf"Asymptotic normality of the Poisson MLE (n={n})")
ax.legend()
plt.show()


---
## Part 3 — The three asymptotic tests: Wald, score, likelihood-ratio

For testing \(H_0:\theta=\theta_0\) versus \(H_1:\theta\ne\theta_0\),
three asymptotically equivalent statistics all converge to
\(\chi^2_1\) under \(H_0\):

- **Likelihood ratio test (LRT)**:
  \(\displaystyle R = -2\log\frac{L(\theta_0)}{\hat{L}_n}
  = 2\big(\ell_n(\hat{\theta}_n)-\ell_n(\theta_0)\big)
  \;\overset{H_0}{\Rightarrow}\;\chi^2_1.\)

- **Score test (Rao)**:
  \(\displaystyle S = \frac{U_n(\theta_0)^2}{I_n(\theta_0)}
  \;\overset{H_0}{\Rightarrow}\;\chi^2_1,\)
  where \(U_n(\theta)=\partial\ell_n/\partial\theta\) is the sample score
  and \(I_n(\theta_0)\) is the Fisher information for the whole sample.

- **Wald test**:
  \(\displaystyle W = \frac{(\hat{\theta}_n-\theta_0)^2}{\widehat{\mathrm{Var}}(\hat{\theta}_n)}
  \;\overset{H_0}{\Rightarrow}\;\chi^2_1.\)

Note that the Wald statistic can also be written as
\(W = n(\hat{\theta}_n-\theta_0)^2\,/\,\hat{V}_n\) with
\(\hat{V}_n\) a consistent estimator of the asymptotic variance of
\(\hat{\theta}_n\). The signless square root of each statistic is
asymptotically **standard normal**.

We implement all three for the Poisson model (where the score and
information have closed forms) and check, by simulating under \(H_0\),
that each statistic's empirical distribution matches \(\chi^2_1\).


In [ ]:
def poisson_loglik(theta, sample):
    '''Poisson log-likelihood for one scalar theta (vectorized over theta).'''
    theta = np.atleast_1d(theta).astype(float)
    return np.sum(sample[None, :] * np.log(theta[:, None]) - theta[:, None], axis=1)

def poisson_score(theta, sample):
    '''U_n(theta) = sum(x_i)/theta - n.'''
    return sample.sum() / theta - len(sample)

def poisson_info(theta, n):
    '''Expected Fisher information for n observations: n / theta.'''
    return n / theta

def three_stats(sample, theta0):
    '''Return (LRT, Score, Wald) for testing H0: theta = theta0.'''
    n = len(sample)
    theta_hat = sample.mean()                          # MLE
    # Log-likelihoods (per-obs sum); LRT uses 2 * (ell(hat) - ell(theta0))
    ell_hat = np.sum(sample * np.log(theta_hat) - theta_hat)
    ell_0   = np.sum(sample * np.log(theta0)   - theta0)
    R = 2.0 * (ell_hat - ell_0)                        # LRT statistic
    # Score test: U_n(theta0)^2 / I_n(theta0)
    U0 = poisson_score(theta0, sample)
    I0 = poisson_info(theta0, n)
    Sstat = (U0 ** 2) / I0
    # Wald test: (theta_hat - theta0)^2 / Var(theta_hat),  Var = theta_hat / n
    Vhat = theta_hat / n
    W = (theta_hat - theta0) ** 2 / Vhat
    return R, Sstat, W

# Simulate under H0 and compare the three statistics to chi-square_1
theta0 = 3.0
n = 200
reps = 40000
rng = np.random.default_rng(101)

R = np.empty(reps); Sstat = np.empty(reps); Wstat = np.empty(reps)
for b in range(reps):
    s = rng.poisson(theta0, size=n)
    R[b], Sstat[b], Wstat[b] = three_stats(s, theta0)

chi2_1 = stats.chi2(df=1)
print(f"H0: theta = {theta0},  n = {n},  reps = {reps}")
print(f"{'stat':>6} {'mean':>8} {'var':>8}  (chi2_1: mean={chi2_1.mean():.3f}, var={chi2_1.var():.3f})")
for name, stat in [("LRT", R), ("Score", Sstat), ("Wald", Wstat)]:
    print(f"{name:>6} {stat.mean():>8.3f} {stat.var():>8.3f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
grid = np.linspace(0, 12, 400)
for ax, name, stat, color in zip(axes, ["LRT (R)", "Score (S)", "Wald (W)"],
                                 [R, Sstat, Wstat],
                                 ["steelblue", "seagreen", "darkorange"]):
    ax.hist(stat, bins=80, density=True, color=color, edgecolor="white",
            alpha=0.75, label="simulated")
    ax.plot(grid, chi2_1.pdf(grid), color="crimson", lw=2.5,
            label=r"$\chi^2_1$")
    ax.set_title(name); ax.set_xlabel("statistic"); ax.legend(fontsize=9)
axes[0].set_ylabel("density")
fig.suptitle(r"All three tests are $\chi^2_1$ under $H_0$ (Poisson, n={})".format(n),
             y=1.03)
plt.tight_layout()
plt.show()


---
## Part 4 — The signed square roots are standard normal, and power under \(H_1\)

Because \(\chi^2_1 = Z^2\) for \(Z\sim\mathcal{N}(0,1)\), the **signed
square root** of each statistic is asymptotically standard normal:

$$
Z_R = \operatorname{sign}(\hat{\theta}_n-\theta_0)\sqrt{R},\quad
Z_S = \frac{U_n(\theta_0)}{\sqrt{I_n(\theta_0)}},\quad
Z_W = \frac{\hat{\theta}_n-\theta_0}{\widehat{SE}(\hat{\theta}_n)}
\;\;\overset{H_0}{\Rightarrow}\;\; \mathcal{N}(0,1).
$$

This is the form used to build two-sided \(z\)-tests and Wald confidence
intervals \(\hat{\theta}_n \pm z_{1-\alpha/2}\,\widehat{SE}\).

We (1) confirm the standard-normal limit of the signed roots under
\(H_0\), and (2) look at **power**: we simulate data from
\(\theta_1 = 3.3\) (a local alternative) and watch the rejection rate at
the 5% level climb above 0.05, with all three tests agreeing closely for
large \(n\) (they are asymptotically equivalent).


In [ ]:
theta0 = 3.0
theta1 = 3.3          # alternative used for the power study
n = 200
reps = 40000
rng = np.random.default_rng(2024)
z_crit = stats.norm.ppf(0.975)   # two-sided 5% critical value

# --- Under H0: signed roots should be N(0,1) ---
Z_R = np.empty(reps); Z_S = np.empty(reps); Z_W = np.empty(reps)
for b in range(reps):
    s = rng.poisson(theta0, size=n)
    R, Sstat, W = three_stats(s, theta0)
    theta_hat = s.mean()
    Z_R[b] = np.sign(theta_hat - theta0) * np.sqrt(R)
    Z_S[b] = np.sign(theta_hat - theta0) * np.sqrt(Sstat)
    Z_W[b] = np.sign(theta_hat - theta0) * np.sqrt(W)

print("Under H0 — signed roots (should be N(0,1)):")
for name, z in [("Z_R", Z_R), ("Z_S", Z_S), ("Z_W", Z_W)]:
    print(f"  {name}: mean={z.mean():+.4f}, var={z.var(ddof=1):.4f}, "
          f"rej@5%={np.mean(np.abs(z) > z_crit):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
grid = np.linspace(-4, 4, 300)
for ax, name, z, color in zip(axes, ["signed sqrt(LRT)", "signed sqrt(Score)", "signed sqrt(Wald)"],
                              [Z_R, Z_S, Z_W],
                              ["steelblue", "seagreen", "darkorange"]):
    ax.hist(z, bins=70, density=True, color=color, edgecolor="white", alpha=0.75)
    ax.plot(grid, stats.norm.pdf(grid), color="crimson", lw=2.5, label=r"$\mathcal{N}(0,1)$")
    ax.axvline(-z_crit, color="black", ls="--", lw=1); ax.axvline(z_crit, color="black", ls="--", lw=1)
    ax.set_title(name); ax.legend(fontsize=9); ax.set_xlabel("z")
axes[0].set_ylabel("density")
fig.suptitle(r"Signed square roots are $\mathcal{N}(0,1)$ under $H_0$", y=1.03)
plt.tight_layout()
plt.show()

# --- Power under the local alternative theta1 ---
ns = [20, 50, 100, 200, 500, 1000]
reps_pow = 8000
rng = np.random.default_rng(99)
power = {"LRT": [], "Score": [], "Wald": []}
for n in ns:
    rej = {k: 0 for k in power}
    for _ in range(reps_pow):
        s = rng.poisson(theta1, size=n)
        R, Sstat, W = three_stats(s, theta0)
        rej["LRT"]   += int(R > stats.chi2.ppf(0.95, df=1))
        rej["Score"] += int(Sstat > stats.chi2.ppf(0.95, df=1))
        rej["Wald"]  += int(W > stats.chi2.ppf(0.95, df=1))
    for k in power:
        power[k].append(rej[k] / reps_pow)

print(f"\nPower at theta1={theta1}, alpha=0.05:")
print(f"{'n':>6} " + " ".join(f"{k:>8}" for k in power))
for i, n in enumerate(ns):
    print(f"{n:>6} " + " ".join(f"{power[k][i]:>8.3f}" for k in power))

fig, ax = plt.subplots(figsize=(9, 5))
for k, c in zip(power, ["steelblue", "seagreen", "darkorange"]):
    ax.plot(ns, power[k], "o-", lw=2, color=c, label=k)
ax.set_xlabel("sample size n"); ax.set_ylabel("power")
ax.set_title(rf"Power grows with $n$ for the three tests ($H_1:\\theta={theta1}$)")
ax.axhline(0.05, color="black", ls="--", lw=1, label="size 0.05")
ax.legend()
plt.show()


---
## Part 5 — Vector parameter: the multivariate Wald / score / LRT

The vector form of the theorem is

$$
\sqrt{n}\,(\hat{\boldsymbol{\theta}}_n-\boldsymbol{\theta}_0)
\;\xRightarrow{d}\;
\mathcal{N}_p\!\big(\boldsymbol{0},\ I(\boldsymbol{\theta}_0)^{-1}\big),
$$

with \(I(\boldsymbol{\theta}_0)\) the \(p\times p\) Fisher information
matrix. The three tests generalize: the LRT is still
\(2(\ell_n(\hat{\theta})-\ell_n(\theta_0))\overset{H_0}{\Rightarrow}\chi^2_p\);
the score test uses the quadratic form
\(U_n(\theta_0)^{\top} I_n(\theta_0)^{-1} U_n(\theta_0)\); the Wald test uses
\(n(\hat{\theta}_n-\theta_0)^{\top}\hat{V}_n^{-1}(\hat{\theta}_n-\theta_0)\).

We illustrate with a 2-parameter model — the **Normal\((\mu,\sigma^2)\)**
with both parameters unknown — and test
\(H_0:(\mu,\sigma^2)=(\mu_0,\sigma^2_0)\). We compute the LRT and Wald
statistics by numerical optimization (`scipy.optimize.minimize`) and
verify both are \(\chi^2_2\) under \(H_0\).


In [ ]:
from scipy.optimize import minimize

def normal_loglik(params, sample):
    mu, sigma = params
    if sigma <= 0:
        return -np.inf
    n = len(sample)
    return -n * np.log(sigma) - 0.5 * np.sum((sample - mu) ** 2) / sigma**2

def normal_neg_loglik(params, sample):
    return -normal_loglik(params, sample)

# True parameter values (these are also H0 for this illustration)
mu0, sigma0 = 2.0, 1.5
n = 150
reps = 20000
rng = np.random.default_rng(13)

chi2_2 = stats.chi2(df=2)
lrt_stat = np.empty(reps); wald_stat = np.empty(reps)
theta0_vec = np.array([mu0, sigma0])

for b in range(reps):
    s = rng.normal(mu0, sigma0, size=n)
    # MLE: mu_hat = mean, sigma_hat = sqrt(mean((x-mu_hat)^2)) (biased MLE)
    mu_hat = s.mean()
    sigma_hat = np.sqrt(((s - mu_hat) ** 2).mean())
    ell_hat = normal_loglik([mu_hat, sigma_hat], s)
    ell_0   = normal_loglik([mu0, sigma0], s)
    lrt_stat[b] = 2.0 * (ell_hat - ell_0)
    # Wald: need estimated covariance of (mu_hat, sigma_hat).
    # For Normal data, the inverse Fisher information per observation is
    # diag(sigma^2, sigma^2/2); for n obs the covariance is that divided by n.
    Vhat = np.diag([sigma_hat**2, sigma_hat**2 / 2.0]) / n
    diff = np.array([mu_hat - mu0, sigma_hat - sigma0])
    wald_stat[b] = diff @ np.linalg.solve(Vhat, diff)

print(f"Normal(mu,sigma^2) joint test, n={n}, reps={reps}")
print(f"{'stat':>6} {'mean':>8} {'var':>8}  (chi2_2: mean={chi2_2.mean():.3f}, var={chi2_2.var():.3f})")
for name, stat in [("LRT", lrt_stat), ("Wald", wald_stat)]:
    print(f"{name:>6} {stat.mean():>8.3f} {stat.var():>8.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
grid = np.linspace(0, 20, 400)
for ax, name, stat, color in zip(axes, ["LRT (p=2)", "Wald (p=2)"],
                                 [lrt_stat, wald_stat],
                                 ["steelblue", "darkorange"]):
    ax.hist(stat, bins=80, density=True, color=color, edgecolor="white", alpha=0.75)
    ax.plot(grid, chi2_2.pdf(grid), color="crimson", lw=2.5, label=r"$\chi^2_2$")
    ax.set_title(name); ax.set_xlabel("statistic"); ax.legend(fontsize=9)
axes[0].set_ylabel("density")
fig.suptitle(r"Joint LRT and Wald for Normal$(\mu,\sigma^2)$ are $\chi^2_2$ under $H_0$", y=1.03)
plt.tight_layout()
plt.show()

# Empirical size at the nominal 5% level
crit = stats.chi2.ppf(0.95, df=2)
print(f"\nEmpirical size at alpha=0.05 (crit={crit:.3f}):")
print(f"  LRT:  {(lrt_stat > crit).mean():.3f}")
print(f"  Wald: {(wald_stat > crit).mean():.3f}")


---
**Your turn:**

- In Part 2, replace the Poisson model with a **Bernoulli\((p_0)\)** model
  (Fisher information \(I(p)=1/[p(1-p)]\)). Pick \(p_0=0.3\). Does the
  standardized MLE \(\sqrt{n}(\hat{p}-p_0)/\sqrt{p_0(1-p_0)}\) still converge
  to \(\mathcal{N}(0,1)\)? Try \(n=30\) vs \(n=500\) — where is the
  approximation worse, and why?
- In Part 3, the three statistics agree for \(n=200\). Reduce \(n\) to 20
  and re-run the \(\chi^2_1\) comparison. Which statistic has the worst
  small-sample match to \(\chi^2_1\)? Which has the best? Does this match
  the textbook folklore that the LRT is usually the middle ground?
- In Part 4, fix \(n=200\) and sweep the alternative \(\theta_1\in\{3.05,
  3.1, 3.2, 3.5, 4.0\}\). Plot the power curve for all three tests on the
  same axes. Are they still indistinguishable, or do small differences
  appear? Where does each one over/under-shoot?
- In Part 5, test only **one** component — \(H_0:\mu=\mu_0\) with
  \(\sigma^2\) unknown (a nuisance parameter) — using the **profile
  likelihood** \(\ell_P(\mu)=\ell_n(\mu,\hat{\sigma}^2(\mu))\). Verify the
  profile LRT is \(\chi^2_1\) under \(H_0\). Compare its rejection rate to
  the 2-df joint test.
- Implement the **score test** for the 2-parameter Normal model from
  Part 5 using the gradient of the (negative) log-likelihood evaluated at
  \((\mu_0,\sigma_0)\) and the Fisher information matrix — confirm it too
  is \(\chi^2_2\) under \(H_0\).


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. For a sample from \(\mathrm{Poisson}(\theta_0)\) with \(\theta_0=4\),
implement the **Wald**, **score**, and **LRT** statistics for
\(H_0:\theta=\theta_0\) from scratch (no `scipy.optimize` needed — the
Poisson MLE and information have closed forms). Simulate 20,000 datasets
of size \(n=100\) under \(H_0\) and report the empirical mean and
variance of each statistic alongside the \(\chi^2_1\) mean (1) and
variance (2).

2. Derive the **signed square root** of each statistic in Problem 1 and
check that, under \(H_0\), it is approximately standard normal. Report the
empirical mean, variance, and two-sided 5% rejection rate (target 0.05)
for each. Which statistic's signed root is closest to calibration at
\(n=100\)?

3. Run a **power simulation** for the three tests at the 5% level for
\(H_0:\theta=4\) versus the local alternative \(\theta_1=4.4\). Use
\(n\in\{30,100,300\}\) and 5,000 replicates per setting. Plot power versus
\(n\) for all three statistics on one figure. Do the three curves
separate at small \(n\)? By \(n=300\) they should nearly coincide —
verify this.

4. For the 2-parameter Normal model from Part 5, implement the **score
test** for \(H_0:(\mu,\sigma^2)=(\mu_0,\sigma^2_0)\). Use the gradient of
the log-likelihood (the score vector) and the Fisher information matrix
\(I_n(\mu_0,\sigma_0)=\mathrm{diag}(n/\sigma_0^2,\,2n/\sigma_0^2)\).
Simulate 10,000 datasets of size \(n=100\) under \(H_0\) and confirm the
statistic \(U^\top I_n^{-1} U\) is approximately \(\chi^2_2\) (report the
empirical mean and variance).

5. Test a **single component** of a vector parameter using the **profile
likelihood**: for Normal data with \((\mu,\sigma^2)\) unknown, test
\(H_0:\mu=\mu_0\) by profiling out \(\sigma^2\), i.e.
\(\hat{\sigma}^2(\mu)=\frac{1}{n}\sum_i(x_i-\mu)^2\). Define the profile LRT
\(R_P=2[\ell_n(\hat{\mu},\hat{\sigma}^2)-\ell_n(\mu_0,\hat{\sigma}^2(\mu_0))]\).
Simulate 10,000 datasets at \(n=100\) under \((\mu_0,\sigma_0)=(0,1)\) and
confirm \(R_P\overset{H_0}{\Rightarrow}\chi^2_1\). Report the empirical mean
and 5% rejection rate.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Wald, score, and LRT for the Poisson model under \(H_0:\theta=4\):

```python
theta0 = 4.0; n = 100; reps = 20000
rng = np.random.default_rng(0)
R = np.empty(reps); S = np.empty(reps); W = np.empty(reps)
for b in range(reps):
    s = rng.poisson(theta0, size=n)
    th = s.mean()
    ell_hat = np.sum(s * np.log(th) - th)
    ell_0   = np.sum(s * np.log(theta0) - theta0)
    R[b] = 2.0 * (ell_hat - ell_0)
    U0 = s.sum() / theta0 - n
    I0 = n / theta0
    S[b] = U0**2 / I0
    Vhat = th / n
    W[b] = (th - theta0)**2 / Vhat
print("LRT  :", R.mean(), R.var())
print("Score:", S.mean(), S.var())
print("Wald :", W.mean(), W.var())
print("chi2_1:", 1.0, 2.0)
```

All three should land close to mean 1 and variance 2; small Monte Carlo
and higher-order-cumulant deviations remain at \(n=100\).

**2.** Signed roots under \(H_0\) should be \(\mathcal{N}(0,1)\):

```python
z = stats.norm.ppf(0.975)
ZR = np.sign(R - 0) * np.sqrt(np.abs(R))     # sign of (theta_hat-theta0)
# better: recompute sign from theta_hat-theta0
for b in range(reps):
    s = rng.poisson(theta0, size=n); th = s.mean()
    # (recompute stored stats with the sign you saved, or store the sign)
# Quick check using stored R (sign unknown for ties) — store sign instead:
import numpy as np
rng = np.random.default_rng(1)
ZR = np.empty(reps); ZS = np.empty(reps); ZW = np.empty(reps)
for b in range(reps):
    s = rng.poisson(theta0, size=n); th = s.mean()
    d = th - theta0
    R_, S_, W_ = three_stats(s, theta0)   # function from Part 3
    ZR[b] = np.sign(d) * np.sqrt(R_)
    ZS[b] = np.sign(d) * np.sqrt(S_)
    ZW[b] = np.sign(d) * np.sqrt(W_)
for name, zstat in [("ZR", ZR), ("ZS", ZS), ("ZW", ZW)]:
    print(name, zstat.mean(), zstat.var(),
          np.mean(np.abs(zstat) > z))
```

Each should be mean \(\approx 0\), variance \(\approx 1\), rejection
\(\approx 0.05\). At \(n=100\) the Wald signed root is usually best
calibrated because it uses the data-driven SE; the score signed root is
slightly left-skewed for small \(n\).

**3.** Power curves for the three tests vs \(n\):

```python
theta0 = 4.0; theta1 = 4.4
ns = [30, 100, 300]; reps = 5000
rng = np.random.default_rng(2)
crit = stats.chi2.ppf(0.95, df=1)
power = {"LRT": [], "Score": [], "Wald": []}
for n in ns:
    rej = {k: 0 for k in power}
    for _ in range(reps):
        s = rng.poisson(theta1, size=n)
        R_, S_, W_ = three_stats(s, theta0)
        rej["LRT"]   += int(R_ > crit)
        rej["Score"] += int(S_ > crit)
        rej["Wald"]  += int(W_ > crit)
    for k in power: power[k].append(rej[k] / reps)

import matplotlib.pyplot as plt
for k, c in zip(power, ["steelblue", "seagreen", "darkorange"]):
    plt.plot(ns, power[k], "o-", color=c, label=k)
plt.axhline(0.05, color="k", ls="--"); plt.xlabel("n"); plt.ylabel("power")
plt.legend(); plt.show()
```

At \(n=30\) the curves separate (Wald and LRT usually highest); by
\(n=300\) they are nearly on top of each other — the asymptotic
equivalence kicks in.

**4.** Score test for the 2-parameter Normal model:

```python
mu0, sigma0 = 2.0, 1.5
n = 100; reps = 10000
rng = np.random.default_rng(3)
I0 = np.diag([n / sigma0**2, 2 * n / sigma0**2])
Sstat = np.empty(reps)
for b in range(reps):
    s = rng.normal(mu0, sigma0, size=n)
    # score vector at (mu0, sigma0): use sigma (not sigma^2) parameterization
    U = np.array([np.sum(s - mu0) / sigma0**2,
                  np.sum((s - mu0)**2 / sigma0**3) - n / sigma0])
    Sstat[b] = U @ np.linalg.solve(I0, U)
print("mean:", Sstat.mean(), "var:", Sstat.var(),
      "(chi2_2: mean=2, var=4)")
# size at 5%
crit2 = stats.chi2.ppf(0.95, df=2)
print("size:", np.mean(Sstat > crit2))
```

The statistic should have mean \(\approx 2\), variance \(\approx 4\), and
size \(\approx 0.05\).

**5.** Profile-likelihood LRT for \(H_0:\mu=\mu_0\) in the Normal model
(\(\sigma^2\) nuisance):

```python
mu0, sigma0 = 0.0, 1.0
n = 100; reps = 10000
rng = np.random.default_rng(4)
Rprof = np.empty(reps)
for b in range(reps):
    s = rng.normal(mu0, sigma0, size=n)
    mu_hat = s.mean()
    sig2_hat = ((s - mu_hat)**2).mean()
    sig2_prof = ((s - mu0)**2).mean()      # profiled sigma^2 at mu0
    ell_hat = -0.5 * n * np.log(2*np.pi*sig2_hat) - 0.5 * n
    ell_0   = -0.5 * n * np.log(2*np.pi*sig2_prof) - 0.5 * n
    Rprof[b] = 2.0 * (ell_hat - ell_0)
print("mean:", Rprof.mean(), "var:", Rprof.var(),
      "(chi2_1: mean=1, var=2)")
crit1 = stats.chi2.ppf(0.95, df=1)
print("size @5%:", np.mean(Rprof > crit1))
```

The profile LRT is \(\chi^2_1\) under \(H_0\): mean \(\approx 1\),
variance \(\approx 2\), rejection rate \(\approx 0.05\). Profiling out
the nuisance parameter recovers one degree of freedom.


</details>
